# resolvers

> Container resolver registry

In [ ]:
#| default_exp resolvers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
from paar.resolvers import (Resolver, DictResolver, ListTupleResolver, SetResolver,
                            ObjectResolver, resolve_for, register_resolver, _RESOLVERS)
class _Obj:
    def __init__(self): self.x=1; self.y='hi'

def test_dict_resolver():
    r = resolve_for({'a':1})
    assert isinstance(r, DictResolver)
    assert r.children({'a':1, 'b':2}) == [("'a'", "['a']", 1), ("'b'", "['b']", 2)]
def test_list_resolver():
    r = resolve_for([10,20])
    assert isinstance(r, ListTupleResolver)
    assert r.children([10,20]) == [('0','[0]',10), ('1','[1]',20)]
def test_tuple_uses_list_resolver():
    assert isinstance(resolve_for((1,2)), ListTupleResolver)
def test_set_resolver():
    r = resolve_for({1})
    assert isinstance(r, SetResolver)
    assert r.children({7}) == [('0','[0]',7)]
def test_object_resolver():
    r = resolve_for(_Obj())
    assert isinstance(r, ObjectResolver)
    assert r.children(_Obj()) == [('x','.x',1), ('y','.y','hi')]
def test_scalars_not_containers():
    assert resolve_for(5) is None and resolve_for('hi') is None and resolve_for(None) is None
def test_register_before_object_catchall():
    class _R(Resolver):
        def can(self, v): return False
    r = _R(); register_resolver(r)
    obj_i = next(i for i,x in enumerate(_RESOLVERS) if isinstance(x, ObjectResolver))
    assert _RESOLVERS.index(r) < obj_i   # inserted ahead of the catch-all
    _RESOLVERS.remove(r)
for t in [test_dict_resolver,test_list_resolver,test_tuple_uses_list_resolver,
          test_set_resolver,test_object_resolver,test_scalars_not_containers,
          test_register_before_object_catchall]: t()

In [ ]:
#| export
class Resolver:
    "Decides if a value is a container and yields its ordered children."
    def can(self, v): raise NotImplementedError
    def children(self, v): raise NotImplementedError   # -> list[(name, step, child)]

class DictResolver(Resolver):
    def can(self, v): return isinstance(v, dict)
    def children(self, v): return [(repr(k), f'[{k!r}]', val) for k,val in v.items()]

class ListTupleResolver(Resolver):
    def can(self, v): return isinstance(v, (list, tuple))
    def children(self, v): return [(str(i), f'[{i}]', x) for i,x in enumerate(v)]

class SetResolver(Resolver):
    "Sets: positional children in a stable order (sorted by repr); path fragment is display-only, not index-addressable."
    def can(self, v): return isinstance(v, (set, frozenset))
    def children(self, v): return [(str(i), f'[{i}]', x) for i,x in enumerate(sorted(v, key=repr))]

class ObjectResolver(Resolver):
    def can(self, v): return hasattr(v, '__dict__') and bool(vars(v))
    def children(self, v): return [(k, f'.{k}', val) for k,val in sorted(vars(v).items())]

_RESOLVERS = [DictResolver(), ListTupleResolver(), SetResolver(), ObjectResolver()]

def register_resolver(r, last=False):
    "Register a resolver; inserted before the catch-all ObjectResolver unless last=True."
    _RESOLVERS.insert(len(_RESOLVERS) if last else len(_RESOLVERS)-1, r)

def resolve_for(v):
    "First resolver that handles v, else None (None => not a container)."
    for r in _RESOLVERS:
        try:
            if r.can(v): return r
        except Exception: pass
    return None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()